# FAQ - empregadados — Busca Semântica com Embeddings + FAISS + Streamlit

Este notebook explica **passo a passo** o projeto e depois mostra o **código completo**.

> Objetivo: construir uma busca semântica (estilo RAG simples) em um FAQ usando **embeddings**, **FAISS** e uma interface **Streamlit**.


## Visão geral da arquitetura

**Fluxo**:

1. `docs.csv` (FAQ) → cada linha é um documento
2. Geramos **embeddings** para cada documento com `sentence-transformers`
3. Criamos um **índice vetorial** com FAISS e inserimos os vetores
4. No app Streamlit, o usuário digita uma pergunta
5. A pergunta vira embedding → FAISS retorna os Top-K documentos mais similares

> Observação: para manter a aula simples, **não usamos chunking** neste projeto: 1 linha do CSV = 1 documento.


## O que são embeddings (na prática)

Um **embedding** é um vetor numérico que representa o significado de um texto.

- Documento: `"Como solicitar reembolso?"` → vetor com, por exemplo, **384 números**
- Pergunta do usuário → outro vetor com **as mesmas 384 dimensões**

A busca semântica funciona comparando vetores e procurando os mais próximos.

### Por que normalizamos os embeddings?
Usamos `normalize_embeddings=True` para que cada vetor tenha norma 1.

- Com vetores normalizados, o **produto interno** (dot product) fica equivalente à **similaridade cosseno**.
- Isso nos permite usar `IndexFlatIP` no FAISS (IP = inner product) como uma busca por **cosine similarity**.


## O que é FAISS (detalhe por detalhe)

**FAISS (Facebook AI Similarity Search)** é uma biblioteca para **busca por similaridade em vetores**.

Você pode pensar assim:

- Você tem **N vetores** (embeddings) dos documentos
- Você tem **1 vetor** da pergunta
- O FAISS retorna os **k vetores mais similares** e seus **scores**

### Tipos de índice no FAISS
- **Flat (exato)**: compara com todos os vetores; simples e ótimo para datasets pequenos
- **Aproximados (escaláveis)**: IVF, HNSW, PQ… (para milhões de vetores)

No nosso projeto usamos:

- `faiss.IndexFlatIP(dim)`
  - `Flat`: exato
  - `IP`: inner product
  - com vetores normalizados: IP ≈ cosine similarity

### “Inserir índices” no FAISS
Aqui “inserir índice” significa **adicionar vetores** ao índice:

```python
index.add(emb)
```

- `emb` tem shape `(N, D)`
- ao adicionar, o FAISS guarda os vetores e associa IDs implícitos: `0..N-1`
- depois, `index.search(query_vector, k)` retorna esses IDs.


## Por que `dim = emb.shape[1]` é tão importante

Quando geramos embeddings, obtemos uma matriz:

- `emb.shape = (N, D)`
  - `N`: número de documentos
  - `D`: dimensão do embedding (ex.: 384)

Então:

```python
dim = emb.shape[1]
```

Isso pega **D** (número de colunas), que é o tamanho de cada vetor.

O FAISS precisa disso para criar o índice com a dimensão correta:

```python
index = faiss.IndexFlatIP(dim)
```

Se `dim` estiver errado, o FAISS dá erro de incompatibilidade (mismatch).



## Entendendo `dim = emb.shape[1]` com exemplos numéricos

### 1) O que é `emb`

Antes dessa linha, temos:

```python
emb = model.encode(texts, normalize_embeddings=True).astype("float32")
```

Aqui, `emb` é uma **matriz de embeddings** (um embedding por documento). Pense assim:

```text
emb = [
  [0.12, -0.44, 0.91, ...],   ← embedding do documento 1
  [0.33,  0.18, -0.52, ...],  ← embedding do documento 2
  [0.04,  0.77, 0.29, ...],   ← embedding do documento 3
]
```

Matematicamente:

```text
emb.shape = (N, D)
```

onde:

- `N` = número de documentos
- `D` = número de dimensões do embedding

No modelo que usamos (`all-MiniLM-L6-v2`), tipicamente:

```text
D = 384
```

Então um exemplo real seria:

```text
emb.shape = (140, 384)
```

Significa:

- 140 documentos
- cada documento representado por um vetor com 384 números

---

### 2) O que é `.shape`

`.shape` é um atributo do array (NumPy) que retorna o formato da matriz.

Exemplo:

```python
emb.shape
```

poderia retornar:

```text
(140, 384)
```

Ou seja:

```text
(linhas, colunas)
```

---

### 3) O que significa `[1]`

Quando fazemos:

```python
emb.shape[1]
```

estamos pegando o **segundo elemento** da tupla de shape.

Se:

```text
emb.shape = (140, 384)
```

então:

```text
emb.shape[0] = 140   # número de documentos (linhas)
emb.shape[1] = 384   # dimensão do embedding (colunas)
```

Logo:

```python
dim = emb.shape[1]
```

resulta em:

```text
dim = 384
```

---

### 4) Por que o FAISS precisa disso

Quando criamos o índice FAISS:

```python
index = faiss.IndexFlatIP(dim)
```

o FAISS precisa saber **quantas dimensões** têm os vetores, porque ele vai criar uma estrutura para armazenar vetores **daquele tamanho**.

Se o embedding tem:

```text
384 dimensões
```

o índice precisa ser criado como:

```text
IndexFlatIP(384)
```

Caso contrário o FAISS não sabe:

- quantos números existem em cada vetor
- como calcular a similaridade corretamente

---

### 5) O que acontece se estiver errado

Se você fizer:

```python
index = faiss.IndexFlatIP(100)
```

mas os vetores tiverem **384 dimensões**, você terá erro de incompatibilidade (mismatch), porque o FAISS espera vetores de tamanho 100, mas recebe 384.

---

### 6) Visualizando o processo completo

**Embeddings (cada um com 384 dimensões):**

```text
doc1 → [0.12, 0.91, 0.44, ...] (384)
doc2 → [0.33, 0.21, 0.77, ...] (384)
doc3 → [0.05, 0.88, 0.63, ...] (384)
```

**Matriz de embeddings:**

```text
emb =

doc1   0.12 0.91 0.44 ...
doc2   0.33 0.21 0.77 ...
doc3   0.05 0.88 0.63 ...
```

Dimensão:

```text
emb.shape = (3, 384)
```

Então:

```text
dim = 384
```

---

### 7) Intuição geométrica

Cada documento vira um **ponto** em um espaço de **384 dimensões**.

O FAISS armazena todos esses pontos.

Quando chega uma pergunta, por exemplo:

```text
"como solicitar reembolso?"
```

ela vira outro vetor (outro ponto). O FAISS calcula:

```text
qual ponto está mais próximo desse vetor?
```

Isso é **busca semântica**.

---

### 8) Resumo da linha

```python
dim = emb.shape[1]
```

significa:

> Pegue o número de dimensões do embedding para criar o índice vetorial corretamente.

Ou, de forma bem simples:

> “Quantos números existem dentro de cada embedding?”


## Chunking: qual técnica foi escolhida?

Neste projeto **não há chunking** por design (aula rápida e estável).

- Unidade de indexação = **1 linha do CSV**
- Isso funciona muito bem para FAQ/tickets/textos curtos.

### Quando chunking faria sentido?
- PDFs, políticas longas, artigos grandes
- Para esses casos, uma técnica simples é chunking por caracteres com overlap, por exemplo:
  - `chunk_size = 1000` caracteres
  - `overlap = 200` caracteres

Se você quiser, dá para evoluir o projeto para chunking com poucas linhas.


# Código do projeto

A seguir estão os arquivos principais do projeto, com comentários.

## 1) `requirements.txt`


In [ ]:
# requirements.txt (conteúdo)
reqs = """streamlit==1.36.0
pandas==2.2.2
pyarrow==16.1.0
sentence-transformers==3.0.1
faiss-cpu==1.8.0.post1
"""
print(reqs)


## 2) `build_index.py` — cria embeddings + índice FAISS

Este script:

1. Lê `docs.csv`
2. Monta o texto final (título + corpo)
3. Gera embeddings
4. Cria o índice FAISS e **insere os vetores** (`index.add`)
5. Salva `data/faiss.index` e `data/meta.parquet`


In [ ]:
# build_index.py (conteúdo)
build_index_py = r'''
from __future__ import annotations
from pathlib import Path
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

def main():
    # 1) Ler o CSV (FAQ)
    df = pd.read_csv("docs.csv")
    df["title"] = df["title"].fillna("").astype(str)
    df["text"] = df["text"].fillna("").astype(str)
    df["source"] = df["source"].fillna("").astype(str)
    df["doc_id"] = df["doc_id"].astype(str)

    # 2) Texto final para embedding (sem chunking: 1 linha = 1 documento)
    texts = (df["title"] + "\n\n" + df["text"]).tolist()

    # 3) Modelo e embeddings (normalizados para usar cosine via inner product)
    model = SentenceTransformer(MODEL_NAME)
    emb = model.encode(texts, show_progress_bar=True, normalize_embeddings=True).astype("float32")

    # 4) Criar índice FAISS (Flat + Inner Product)
    dim = emb.shape[1]              # D = dimensão do embedding
    index = faiss.IndexFlatIP(dim)  # IP com vetores normalizados ≈ cosine similarity

    # 5) Inserir vetores no índice (a “indexação” acontece aqui)
    index.add(emb)                  # IDs implícitos: 0..N-1

    # 6) Salvar índice e metadados (mesma ordem => meta.iloc[id] bate com o FAISS)
    faiss.write_index(index, str(DATA_DIR / "faiss.index"))
    df.to_parquet(DATA_DIR / "meta.parquet", index=False)

    print("OK: índice salvo em data/faiss.index e metadata em data/meta.parquet")

if __name__ == "__main__":
    main()
'''
print(build_index_py)


## 3) `app.py` — interface Streamlit + busca no FAISS

O app faz:

- carrega o índice e o meta
- transforma a pergunta em embedding
- executa `index.search(query_embedding, k)`
- usa os IDs retornados para buscar os textos no `meta.parquet`
- renderiza os resultados (título, score e resposta)


In [ ]:
# app.py (conteúdo)
app_py = r'''
from __future__ import annotations
from pathlib import Path
import pandas as pd
import streamlit as st
import faiss
from sentence_transformers import SentenceTransformer

DATA_DIR = Path("data")
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

@st.cache_resource
def load_model():
    # Carrega o modelo apenas uma vez
    return SentenceTransformer(MODEL_NAME)

@st.cache_resource
def load_store():
    # Carrega índice FAISS e metadados apenas uma vez
    idx_path = DATA_DIR / "faiss.index"
    meta_path = DATA_DIR / "meta.parquet"
    if not idx_path.exists() or not meta_path.exists():
        raise FileNotFoundError("Rode `python build_index.py` antes.")
    index = faiss.read_index(str(idx_path))
    meta = pd.read_parquet(meta_path)
    return index, meta

def retrieve(query: str, k: int, source_filter: str | None = None):
    model = load_model()
    index, meta = load_store()

    # Opcional: filtro por categoria (source).
    # Para manter simples, aqui recriamos um índice temporário quando filtra.
    if source_filter and source_filter != "todas":
        meta_f = meta[meta["source"] == source_filter].reset_index(drop=True)
        if meta_f.empty:
            return []

        texts = (meta_f["title"].fillna("") + "\n\n" + meta_f["text"].fillna("")).tolist()
        emb = model.encode(texts, normalize_embeddings=True).astype("float32")

        dim = emb.shape[1]
        tmp = faiss.IndexFlatIP(dim)
        tmp.add(emb)

        q = model.encode([query], normalize_embeddings=True).astype("float32")
        scores, ids = tmp.search(q, k)

        results = []
        for score, idx in zip(scores[0], ids[0]):
            if idx == -1:
                continue
            row = meta_f.iloc[int(idx)].to_dict()
            row["score"] = float(score)
            results.append(row)
        return results

    # Caso padrão: busca no índice global já salvo
    q = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        row = meta.iloc[int(idx)].to_dict()
        row["score"] = float(score)
        results.append(row)
    return results

st.set_page_config(page_title="FAQ - empregadados (Busca Semântica)", layout="wide")
st.title("FAQ - empregadados — Busca Semântica (FAISS + Streamlit)")
st.caption("Digite uma pergunta. O sistema recupera as melhores respostas do FAQ usando embeddings e FAISS.")

index, meta = load_store()
sources = ["todas"] + sorted([s for s in meta["source"].dropna().unique().tolist() if str(s).strip()])

with st.sidebar:
    st.header("Config")
    k = st.slider("Top-K", 3, 15, 5)
    source_filter = st.selectbox("Filtrar por categoria (source)", sources, index=0)
    st.caption(f"Modelo: {MODEL_NAME}")

q = st.text_input("Pergunta:", placeholder="Ex.: como solicitar reembolso? qual SLA do suporte? LGPD exclusão de dados?")

if q:
    hits = retrieve(q, k=k, source_filter=source_filter)

    st.subheader("Melhores respostas encontradas")
    if not hits:
        st.warning("Nenhum resultado encontrado para esse filtro. Tente 'todas' ou outra categoria.")
    for i, h in enumerate(hits, 1):
        st.markdown(f"### {i}. {h.get('title','(sem título)')} — score `{h['score']:.4f}`")
        st.caption(f"doc_id: {h.get('doc_id')} | categoria: {h.get('source')}")
        st.write(h.get("text", ""))
        st.divider()

    if hits:
        st.subheader("Resposta sugerida (heurística simples)")
        answer = "\n\n".join([hits[j]["text"] for j in range(min(2, len(hits)))])
        st.write(answer)
else:
    st.info("Exemplos: “como solicitar reembolso?”, “como emitir nota fiscal?”, “como ativar 2FA?”, “qual SLA do suporte?”")
'''
print(app_py)


## Como executar fora do notebook

No terminal, dentro da pasta do projeto:

```bash
pip install -r requirements.txt
python build_index.py
streamlit run app.py
```

- `build_index.py` precisa rodar primeiro para gerar `data/faiss.index` e `data/meta.parquet`.
- Depois o Streamlit abre a interface no navegador.
